## Sentence Transformer

In [1]:
# 1. Load pretrained model
from sentence_transformers import SentenceTransformer
import json

model = SentenceTransformer('all-MiniLM-L6-v2')
model2 = "text-embedding-3-small"

/Users/rayan/learning/git/agentic-rag/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10776.09it/s]


In [2]:
# load json dataset
with open('recipes_dataset.json', 'r') as f:
    data = json.load(f)

print(len(data))
print(data[0])

55
{'id': 1, 'name': 'Classic Margherita Pizza', 'cuisine': 'Italian', 'ingredients': ['2 cups all-purpose flour', '1 teaspoon instant yeast', '1 teaspoon salt', '3/4 cup warm water', '1 tablespoon olive oil', '1/2 cup tomato sauce', '1 cup fresh mozzarella cheese', 'Fresh basil leaves', 'Olive oil for drizzling'], 'instructions': 'Mix flour, yeast, salt, and water to form a dough. Knead for 10 minutes, then let rise for 1 hour. Roll out dough and spread tomato sauce. Add mozzarella cheese and fresh basil. Bake at 475°F (245°C) for 12-15 minutes. Drizzle with olive oil before serving.', 'cooktime_minutes': 65, 'nutrition': {'calories': 320, 'protein': 12, 'carbohydrates': 45, 'fat': 14, 'fiber': 3, 'sodium': 620}}


In [3]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

openai = OpenAI()

def get_metadata(recipe):
    return {
        # "id": recipe['id'],
        "name": recipe['name'],
        "cooktime_minutes": recipe['cooktime_minutes'],
        "cuisine": recipe['cuisine'],
        "nutrition": json.dumps(recipe['nutrition']),
        "ingredients": ", ".join(recipe['ingredients']),
        # "instructions": recipe['instructions']
    }

embeddings = []
ids=[]
documents = []
metadata = []

for recipe in data:
    ids.append(f"id_{recipe['id']}")
    text_to_encode = f"recipe name: {recipe['name']}; cooktime in minutes: {recipe['cooktime_minutes']}; cuisine: {recipe['cuisine']}; nutrition info: {', '.join(f'{k}- {v}' for k, v in recipe['nutrition'].items())}; ingredients: {', '.join(recipe['ingredients'])}; instructions: {recipe['instructions']}"
    documents.append(text_to_encode)
    # embedding = model.encode(text_to_encode).tolist()
    embedding = openai.embeddings.create(input=text_to_encode, model=model2).data[0].embedding
    embeddings.append(embedding)
    metadata.append(get_metadata(recipe))



In [4]:
# print(json.dumps(embeddings[0], indent=2))
print(len(embeddings[0]))
print(len(json.dumps(embeddings[0], indent=2)))
# print(json.dumps(embeddings[0], indent=2)[:500])

1536
34126


In [5]:
# initialize chroma client
import chromadb

client = chromadb.Client()
if 'recipe-embeddings' in [collection.name for collection in client.list_collections()]:
    client.delete_collection('recipe-embeddings')
    print(f"Deleted existing collection 'recipe-embeddings'.")

collection = client.create_collection('recipe-embeddings')
print(f"Created new collection 'recipe-embeddings'.")

print ([collection.name for collection in client.list_collections()])
# add embeddings to collection
collection.add(
    ids=ids,
    embeddings=embeddings,
    documents=documents,
    metadatas=metadata
)

print(f"Added {len(embeddings)} embeddings with average {len(embeddings[0])} dimensions for {len(documents)} documents to collection 'recipe-embeddings'.")

Created new collection 'recipe-embeddings'.
['recipe-embeddings']
Added 55 embeddings with average 1536 dimensions for 55 documents to collection 'recipe-embeddings'.


In [6]:
user_input = "I want a Indian cuisine made with rice."
# encode user input
# user_embedding = model.encode(user_input).tolist()
user_embedding = openai.embeddings.create(input=user_input, model=model2).data[0].embedding

retrieval_results = collection.query(
    query_embeddings=[user_embedding],
    n_results=10,
    # where={
    #     "$and": [
    #         {"cuisine": {"$eq": "Indian"}},
    #         {"has_rice": {"$eq": True}}
    #     ]
    # }
)
retrieved_documents = retrieval_results['documents'][0]
print("Retrieved documents:")
for i, doc in enumerate(retrieved_documents):
    print(f"{i+1}. {doc}")

# Also print metadata to verify filtering worked
if retrieval_results['metadatas'] and retrieval_results['metadatas'][0]:
    print("\nMetadata of retrieved recipes:")
    for i, meta in enumerate(retrieval_results['metadatas'][0]):
        print(f"{i+1}. {meta['name']}; Cuisine: {meta['cuisine']}; Ingredients: {meta['ingredients']}")

Retrieved documents:
1. recipe name: Biryani; cooktime in minutes: 90; cuisine: Indian; nutrition info: calories- 520, protein- 28, carbohydrates- 55, fat- 20, fiber- 3, sodium- 680; ingredients: 2 cups basmati rice, 1 lb chicken thighs, 1 cup yogurt, 2 onions, thinly sliced and fried, 2 tomatoes, chopped, 2 tablespoons biryani masala, 1 teaspoon saffron soaked in 2 tbsp milk, 4 tablespoons ghee, Fresh mint leaves, Fresh cilantro, Whole spices (cardamom, cloves, cinnamon); instructions: Marinate chicken in yogurt and biryani masala for 2 hours. Parboil rice with whole spices until 70% cooked. Layer chicken at the bottom of a pot, add tomatoes, herbs, and fried onions. Top with rice, drizzle saffron milk and ghee. Seal with lid and cook on low heat for 30 minutes. Mix gently before serving.
2. recipe name: Chickpea Curry (Chana Masala); cooktime in minutes: 30; cuisine: Indian; nutrition info: calories- 280, protein- 12, carbohydrates- 38, fat- 10, fiber- 10, sodium- 480; ingredients: 2

In [16]:
# Cross Encoder
from sentence_transformers import CrossEncoder
import torch

# This encoder gave ImportError 
# cross_encoder_model = CrossEncoder('Qwen/Qwen3-VL-Reranker-2B')

# This encoder wasn't giving proper ranking
# cross_encoder_model = CrossEncoder('cross-encoder/stsb-distilroberta-base')

cross_encoder_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

query = user_input
pairs = [(query, doc) for doc in retrieved_documents]
cross_encoder_scores = cross_encoder_model.predict(pairs, activation_fn=torch.nn.Sigmoid(), prompt="Retrieve images or text relevant to users' query")  # Use sigmoid activation for scores between 0 and 1

#rank the scores and documents based on the scores
ranked_indices = sorted(range(len(cross_encoder_scores)), key=lambda i: cross_encoder_scores[i], reverse=True)
sorted_documents = [retrieved_documents[i] for i in ranked_indices]

print("\nSorted documents based on Cross Encoder scores:\n")
print(f"Query : {query}")
for i, doc in enumerate(sorted_documents):
    print(f"{cross_encoder_scores[ranked_indices[i]]:.5f} : {doc} ")


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 12643.24it/s]



Sorted documents based on Cross Encoder scores:

Query : I want a Indian cuisine made with rice.
0.00172 : recipe name: Biryani; cooktime in minutes: 90; cuisine: Indian; nutrition info: calories- 520, protein- 28, carbohydrates- 55, fat- 20, fiber- 3, sodium- 680; ingredients: 2 cups basmati rice, 1 lb chicken thighs, 1 cup yogurt, 2 onions, thinly sliced and fried, 2 tomatoes, chopped, 2 tablespoons biryani masala, 1 teaspoon saffron soaked in 2 tbsp milk, 4 tablespoons ghee, Fresh mint leaves, Fresh cilantro, Whole spices (cardamom, cloves, cinnamon); instructions: Marinate chicken in yogurt and biryani masala for 2 hours. Parboil rice with whole spices until 70% cooked. Layer chicken at the bottom of a pot, add tomatoes, herbs, and fried onions. Top with rice, drizzle saffron milk and ghee. Seal with lid and cook on low heat for 30 minutes. Mix gently before serving. 
0.00064 : recipe name: Chickpea Curry (Chana Masala); cooktime in minutes: 30; cuisine: Indian; nutrition info: ca

In [ ]:
# Sparse retrieval
from sentence_transformers import SparseEncoder, util

user_query = ["I want a Indian cuisine made with rice.", "which rice dish has the least calories?", "what dessert can I make in least amount of time?"]

#1. load pretrained sparse encoder model
encoder = SparseEncoder('naver/splade-cocondenser-ensembledistil')

#2. encode the query
query_embedding = encoder.encode_query(user_query, prompt_name="query")
print(f"Query embedding shape: {query_embedding.shape}")

document_embeddings = encoder.encode_document(documents, prompt_name="document")
print(f"Document embeddings shape: {document_embeddings.shape}")

#4. perform semantic search using the query embedding and document embeddings
top_k=min(5, len(documents))
results = util.semantic_search(query_embedding, document_embeddings, top_k=top_k, score_function=encoder.similarity)

#5. sort the results based on the scores and print the top-k results
for i, query_results in enumerate(results):
    print(f"\nQuery: {user_query[i]}")
    for j, result in enumerate(query_results):
        doc_id = result['corpus_id']
        score = result['score']
        print(f"Rank {j+1}: Document ID: {doc_id}, Score: {score:.4f}, Document: {documents[doc_id]}")


Loading weights: 100%|██████████| 204/204 [00:00<00:00, 45587.83it/s]


Query embedding shape: torch.Size([3, 30522])
Document embeddings shape: torch.Size([55, 30522])

Query: I want a Indian cuisine made with rice.
Rank 1: Document ID: 50, Score: 15.3027, Document: recipe name: Biryani; cooktime in minutes: 90; cuisine: Indian; nutrition info: calories- 520, protein- 28, carbohydrates- 55, fat- 20, fiber- 3, sodium- 680; ingredients: 2 cups basmati rice, 1 lb chicken thighs, 1 cup yogurt, 2 onions, thinly sliced and fried, 2 tomatoes, chopped, 2 tablespoons biryani masala, 1 teaspoon saffron soaked in 2 tbsp milk, 4 tablespoons ghee, Fresh mint leaves, Fresh cilantro, Whole spices (cardamom, cloves, cinnamon); instructions: Marinate chicken in yogurt and biryani masala for 2 hours. Parboil rice with whole spices until 70% cooked. Layer chicken at the bottom of a pot, add tomatoes, herbs, and fried onions. Top with rice, drizzle saffron milk and ghee. Seal with lid and cook on low heat for 30 minutes. Mix gently before serving.
Rank 2: Document ID: 53, Sc

In [108]:
# Test different filter approaches
results = collection.get(include=['metadatas'])

# Check exact ingredients values for rice recipes
print("Indian recipes with rice in ingredients:")
for meta in results['metadatas']:
    if meta['cuisine'] == 'Indian' and 'rice' in meta['ingredients'].lower():
        print(f'Recipe: {meta["name"]}')
        print(f'  Ingredients: {meta["ingredients"]}')
        print()

# Test exact match filter
print("=== Test: Exact match for 'rice' ===")
retrieval_results = collection.query(
    query_embeddings=[user_embedding],
    n_results=5,
    where={"ingredients": {"$eq": "2 cups basmati rice, 1 lb chicken thighs, 1 cup yogurt, 2 onions, thinly sliced and fried, 2 tomatoes, chopped, 2 tablespoons biryani masala, 1 teaspoon saffron soaked in 2 tbsp milk, 4 tablespoons ghee, Fresh mint leaves, Fresh cilantro, Whole spices (cardamom, cloves, cinnamon)"}}
)
retrieved_documents = retrieval_results['documents'][0]
print(f"Results: {len(retrieved_documents)}")
for i, doc in enumerate(retrieved_documents):
    print(f"{i+1}. {doc[:100]}...")

# Test with $contains on a simpler string
print("\n=== Test: $contains 'basmati' ===")
retrieval_results = collection.query(
    query_embeddings=[user_embedding],
    n_results=5,
    where={"ingredients": {"$contains": "basmati"}}
)
retrieved_documents = retrieval_results['documents'][0]
print(f"Results: {len(retrieved_documents)}")
for i, doc in enumerate(retrieved_documents):
    print(f"{i+1}. {doc[:100]}...")

# Test with $contains 'Naan'
print("\n=== Test: $contains 'Naan' ===")
retrieval_results = collection.query(
    query_embeddings=[user_embedding],
    n_results=5,
    where={"ingredients": {"$contains": "Naan"}}
)
retrieved_documents = retrieval_results['documents'][0]
print(f"Results: {len(retrieved_documents)}")
for i, doc in enumerate(retrieved_documents):
    print(f"{i+1}. {doc[:100]}...")

# Test with $contains 'rice'
print("\n=== Test: $contains 'rice' ===")
retrieval_results = collection.query(
    query_embeddings=[user_embedding],
    n_results=5,
    where={"ingredients": {"$contains": "rice"}}
)
retrieved_documents = retrieval_results['documents'][0]
print(f"Results: {len(retrieved_documents)}")
for i, doc in enumerate(retrieved_documents):
    print(f"{i+1}. {doc[:100]}...")

# Test combined filter with $contains 'rice'
print("\n=== Test: Combined filter with $contains 'rice' ===")
retrieval_results = collection.query(
    query_embeddings=[user_embedding],
    n_results=5,
    where={
        "$and": [
            {"cuisine": {"$eq": "Indian"}},
            {"ingredients": {"$contains": "rice"}}
        ]
    }
)
retrieved_documents = retrieval_results['documents'][0]
print(f"Results: {len(retrieved_documents)}")
for i, doc in enumerate(retrieved_documents):
    print(f"{i+1}. {doc[:100]}...")

Indian recipes with rice in ingredients:
Recipe: Biryani
  Ingredients: 2 cups basmati rice, 1 lb chicken thighs, 1 cup yogurt, 2 onions, thinly sliced and fried, 2 tomatoes, chopped, 2 tablespoons biryani masala, 1 teaspoon saffron soaked in 2 tbsp milk, 4 tablespoons ghee, Fresh mint leaves, Fresh cilantro, Whole spices (cardamom, cloves, cinnamon)

Recipe: Chickpea Curry (Chana Masala)
  Ingredients: 2 cans chickpeas, drained, 1 onion, diced, 2 tomatoes, pureed, 3 cloves garlic, minced, 1-inch ginger, grated, 1 teaspoon cumin seeds, 1 teaspoon coriander powder, 1 teaspoon garam masala, 1/2 teaspoon turmeric, 2 tablespoons vegetable oil, Fresh cilantro, Naan or rice

=== Test: Exact match for 'rice' ===
Results: 1
1. recipe name- Biryani; cooktime in minutes- 90; cuisine- Indian; nutrition info- calories: 520, prote...

=== Test: $contains 'basmati' ===
Results: 0

=== Test: $contains 'Naan' ===
Results: 0

=== Test: $contains 'rice' ===
Results: 0

=== Test: Combined filter with $co

## Data Visualization

In [ ]:
cuisines = set(recipe['cuisine'] for recipe in data)
print("Available cuisines:")
for cuisine in cuisines:
    print(f"- {cuisine}")

In [132]:
# 3D Visualization of Chroma Vector Store using t-SNE
import plotly.graph_objects as go
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import numpy as np

# Get embeddings from collection
results = collection.get(include=['documents', 'embeddings'])
embeddings = np.array(results['embeddings'])
documents = results['documents']
ids = results['ids']

# Extract cuisine labels for coloring
cuisines = []
for doc in documents:
    if 'cuisine:' in doc:
        cuisine = doc.split('cuisine:')[1].split(';')[0].strip()
        cuisines.append(cuisine)
    else:
        cuisines.append('Unknown')

# Reduce to 3D using t-SNE (better for visualization) or PCA (faster)
print("Reducing dimensions with t-SNE...")
tsne = TSNE(n_components=3, random_state=42, perplexity=min(30, len(embeddings)-1))
embeddings_3d = tsne.fit_transform(embeddings)

# Alternative: PCA (uncomment to use instead)
# print("Reducing dimensions with PCA...")
# pca = PCA(n_components=3)
# embeddings_3d = pca.fit_transform(embeddings)

# Create color mapping for cuisines
unique_cuisines = list(set(cuisines))
color_map = {cuisine: i for i, cuisine in enumerate(unique_cuisines)}
colors = [color_map[c] for c in cuisines]

# Create hover text with recipe info
hover_texts = []
for i, (doc, id_) in enumerate(zip(documents, ids)):
    # Extract key info for hover
    name = doc.split('recipe name:')[1].split(';')[0].strip() if 'recipe name:' in doc else 'Unknown'
    cuisine = cuisines[i]
    cooktime = doc.split('cooktime in minutes:')[1].split(';')[0].strip() if 'cooktime in minutes:' in doc else 'Unknown'
    hover_texts.append(f"ID: {id_}<br>Name: {name}<br>Cuisine: {cuisine}<br>Cook time: {cooktime} min")

# Create 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=embeddings_3d[:, 0],
    y=embeddings_3d[:, 1],
    z=embeddings_3d[:, 2],
    mode='markers',
    marker=dict(
        size=6,
        color=colors,
        colorscale='Viridis',
        showscale=True,
        colorbar=dict(title="Cuisine Index"),
        opacity=0.8
    ),
    text=hover_texts,
    hoverinfo='text',
    name='Recipes'
)])

fig.update_layout(
    title='3D Visualization of Recipe Embeddings (t-SNE)',
    scene=dict(
        xaxis_title='t-SNE 1',
        yaxis_title='t-SNE 2',
        zaxis_title='t-SNE 3'
    ),
    width=900,
    height=700,
    margin=dict(l=0, r=0, b=0, t=40)
)

fig.show()

# Also create a 2D version for comparison
print("\nCreating 2D visualization...")
tsne_2d = TSNE(n_components=2, random_state=42, perplexity=min(30, len(embeddings)-1))
embeddings_2d = tsne_2d.fit_transform(embeddings)

fig2d = go.Figure(data=[go.Scatter(
    x=embeddings_2d[:, 0],
    y=embeddings_2d[:, 1],
    mode='markers',
    marker=dict(
        size=8,
        color=colors,
        colorscale='Viridis',
        showscale=True,
        colorbar=dict(title="Cuisine Index"),
        opacity=0.8
    ),
    text=hover_texts,
    hoverinfo='text'
)])

fig2d.update_layout(
    title='2D Visualization of Recipe Embeddings (t-SNE)',
    xaxis_title='t-SNE 1',
    yaxis_title='t-SNE 2',
    width=800,
    height=600
)

fig2d.show()

Reducing dimensions with t-SNE...



Creating 2D visualization...
